# M2_E4: multi-pass blocking + LightGBM

This notebook orchestrates the reusable `experiments.M2_E4.run_experiment` implementation. It evaluates only training data: Source 1 entities are split into train/validation strata by match count using a deterministic SHA-256 ordering with a 20% holdout and seed 42. Every S2/S3 training record is indexed in a temporary disk-backed SQLite inverted index; LightGBM is fitted on a reproducible, stratified cap of train-fold Source 1 entities with deterministically sampled/weighted negative candidates. The test split is not loaded or used for threshold selection.

The candidate files produced here describe the validation fold, not the hidden-label test submission.

In [ ]:
from pathlib import Path
from experiments.M2_E4.run_experiment import ExperimentConfig, run_experiment

PROJECT_ROOT = Path.cwd()
config = ExperimentConfig(
    train_dir=PROJECT_ROOT / "student_resource" / "dataset" / "train",
    output_dir=PROJECT_ROOT / "experiments" / "M2_E4",
    validation_fraction=0.20,
    random_seed=42,
    max_train_entities=25_000,
    negative_samples_per_entity=30,
    chunk_size=20_000,
    max_bucket_size=100,
    prediction_threshold=0.5,
)
metrics = run_experiment(config)
{key: metrics[key] for key in (
    "validation_source1_rows", "candidate_pair_count", "candidate_count_average",
    "candidate_count_median", "candidate_count_p95", "candidate_count_max",
    "candidate_reduction_ratio", "blocking_recall", "matching_precision_micro",
    "matching_recall_micro", "matching_precision_macro", "matching_recall_macro",
    "macro_f0_5", "singleton_accuracy", "runtime_seconds",
)}

Outputs are written to `experiments/M2_E4/candidate_pairs.tsv`, `experiments/M2_E4/matching_results.tsv`, `experiments/M2_E4/metrics.json`, and `experiments/experiment_results.csv`. These are train-validation evaluation artifacts; do not upload them as test submissions.